# 🔄 Usecase 5: The "Closed Loop" — Conversational Analytics & Bottleneck Discovery

Based on the Google Cloud technical blog [*The "Closed Loop" for Agent Observability and Analysis: Connecting ADK, BigQuery, and Conversational Analytics*](https://medium.com/google-cloud/the-closed-loop-for-agent-observability-and-analysis-connecting-adk-bigquery-and-d8fe54971b35), this notebook demonstrates how to query your `agent_events` trace telemetry in natural language to discover common user intents, identify bottleneck tools, and generate executive summaries (`client.insights()`).

```mermaid
flowchart LR
    A[Raw Agent Telemetry in BigQuery] --> B[SDK client.insights Analysis]
    B --> C[Cluster Common User Intents]
    B --> D[Isolate Bottleneck Tools]
    B --> E[Synthesize Executive Report]
    C --> F[Continuous Agent Product Improvement]
    D --> F
```

### Key Derived Metrics & Capabilities:
- **Bottleneck Analysis**: Identifying slow tools, timeouts, and recurring exceptions.
- **Intent Clustering**: Summarizing the top tasks and questions users ask your agents.
- **Natural Language SQL Analytics**: Querying `agent_events` with Gemini conversational analytics.

In [1]:
import os
from google.auth import default
from google.cloud import bigquery
from bigquery_agent_analytics import Client

credentials, _ = default()
PROJECT_ID = "nikunjbhartia-test-clients"
DATASET_ID = "agent_analytics"
TABLE_ID = "agent_events"
BQ_LOCATION = "asia-southeast1"

client = Client(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=TABLE_ID,
    location=BQ_LOCATION,
)
bq_client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
print("✅ SDK Client connected for Closed-Loop Conversational Analytics.")


/Users/nikunjbhartia/Desktop/projects/agents/lineage-agent/.venv/lib/python3.14/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


/Users/nikunjbhartia/Desktop/projects/agents/lineage-agent/.venv/lib/python3.14/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


✅ SDK Client connected for Closed-Loop Conversational Analytics.


## 1) Automated Executive Summary & Bottleneck Discovery (`client.insights()`)

The SDK automatically inspects recent agent traces to synthesize an executive summary of user interactions, latency distribution, and tool bottlenecks.

In [2]:
traces = client.list_traces()
try:
    report = client.insights()
    summary = getattr(report, "summary", None)
    if callable(summary):
        print(summary())
    else:
        print(str(report))
except Exception as e:
    print(f"LLM Insights note: {e}")
    print("\n--- CLOSED-LOOP AUTOMATED TRACE SUMMARY ---")
    total_sessions = len(traces)
    llm_calls = 0
    tool_calls = 0
    err_spans = 0
    for t in traces:
        for s in getattr(t, "spans", []):
            st = getattr(s, "span_type", "")
            if st == "LLM_REQUEST":
                llm_calls += 1
            elif st in ("TOOL_COMPLETED", "TOOL_STARTING"):
                tool_calls += 1
            if "ERROR" in st or getattr(s, "error", None):
                err_spans += 1
    print(f"• Total Analyzed Sessions : {total_sessions}")
    print(f"• Total LLM Invocations   : {llm_calls}")
    print(f"• Total Tool Executions   : {tool_calls}")
    print(f"• Total Error Spans       : {err_spans}")
    if total_sessions > 0:
        print(f"• Overall Success Rate    : {((total_sessions - err_spans) / total_sessions) * 100:.1f}%")


LLM Insights note: No API key was provided. Please pass a valid API key. Learn how to create an API key at https://ai.google.dev/gemini-api/docs/api-key.

--- CLOSED-LOOP AUTOMATED TRACE SUMMARY ---
• Total Analyzed Sessions : 4
• Total LLM Invocations   : 0
• Total Tool Executions   : 0
• Total Error Spans       : 0
• Overall Success Rate    : 100.0%


## 2) Conversational Analytics: Natural Language to SQL Queries over `agent_events`

Because `agent_events` has a standardized schema, product teams and SREs can ask natural language questions about agent behavior using BigQuery AI or conversational BI tools.

### Example Conversational Queries:
- *"Which tools failed most frequently over the last 24 hours?"*
- *"What is the average token consumption per session for lineage_agent?"*
- *"Show me the top 5 slowest agent interactions this week."*

In [3]:
# Example: Translating "Which tools failed most frequently?" into BigQuery SQL
query_nl = f"""
    SELECT
        COALESCE(tool_name, 'unknown_tool') AS Tool_Name,
        COUNT(1) AS Total_Executions,
        COUNTIF(status = 'ERROR' OR error_message IS NOT NULL) AS Failed_Executions
    FROM `{PROJECT_ID}.{DATASET_ID}.v_tool_completed`
    GROUP BY Tool_Name
    ORDER BY Failed_Executions DESC
"""
df_nl = bq_client.query(query_nl).to_dataframe()
display(df_nl)


,Tool_Name,Total_Executions,Failed_Executions
0,run_lineage_extraction,1,0
1,run_bigquery_sql,9,0
2,inspect_table_schema,1,0
3,list_bigquery_datasets,1,0
